# 2-VM Big Data Cluster Reproduction Runbook

Complete setup and verification guide for the **NASA EONET v3 + NASA POWER** pipeline running on `master` (`192.168.56.20`) and `worker` (`192.168.56.21`).

## 1. Architecture & Resource Allocation

| Hostname | Static IP | Specs | Services & Daemons |
| :--- | :--- | :--- | :--- |
| `master` | `192.168.56.20` | 4 vCPU, 10 GB RAM, 50 GB Disk | HDFS NameNode, YARN ResourceManager, **DataNode 1, NodeManager 1**, Kafka 4.4.0 (KRaft), Python Producer |
| `worker` | `192.168.56.21` | 4 vCPU, 10 GB RAM, 50 GB Disk | **DataNode 2, NodeManager 2**, PostgreSQL 16 + TimescaleDB, Grafana |


## 2. Network, SSH & Base Runtime Setup

1. Attach **Adapter 1 (NAT)** and **Adapter 2 (Host-Only `192.168.56.0/24`)** to both Ubuntu 22.04 LTS VMs.
2. Apply `setup/network/netplan-master.yaml` on `master` and `setup/network/netplan-worker.yaml` on `worker` (`sudo netplan apply`).
3. Copy `setup/network/hosts.template` to `/etc/hosts` on both VMs and enable `chrony` with UTC timezone.
4. Install `openssh-server` on both VMs, generate an Ed25519 key on `master`, and run `ssh-copy-id ubuntu@master` and `ssh-copy-id ubuntu@worker`.
5. Install `openjdk-17-jdk` and `python3-venv` on both VMs and copy `setup/network/bigdata-env.sh` to `/etc/profile.d/bigdata.sh`.

## 3. Service Configuration Deployment

- **Kafka 4.4.0 (KRaft on `master`):** Copy `setup/kafka/server.properties` to `/opt/kafka/config/server.properties`, format storage with `kafka-storage.sh format --standalone`, install `setup/systemd/kafka.service`, and create topic `nasa-eonet-raw` (`--partitions 4 --replication-factor 1`).
- **Hadoop 3.4.3 & Spark on YARN (both VMs):** Copy `setup/hadoop/*` to `/opt/hadoop/etc/hadoop/` and `setup/spark/spark-defaults.conf` to `/opt/spark/conf/`, format NameNode once, and start DFS (`RF=2`) + YARN.
- **Python Producer (`master`):** Create a local `.env` from `.env.example` (kept out of Git), install `requirements.txt` inside `venv/`, and enable `setup/systemd/eonet-producer.service`.
- **TimescaleDB & Grafana (`worker`):** Install PostgreSQL 16 + TimescaleDB, run `setup/sql/01_init_timescaledb.sql`, allow `192.168.56.0/24` in `pg_hba.conf`, and enable `grafana-server`.

## 4. Live Cluster Verification

Execute the cell below on `master` to verify all daemons, HDFS replication nodes, YARN workers, and Kafka topic partitions.

In [ ]:
%%bash
bash ~/Bigdata-NASA-Eonet-Pipeline/setup/scripts/health_check.sh
